In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rohitmali28/bestcodingplatform/BestCodingPlatform.csv


In [10]:
df=pd.read_csv('/kaggle/input/datasets/rohitmali28/bestcodingplatform/BestCodingPlatform.csv')

In [12]:
platform_profiles = df.groupby("Platform").agg({
    "Active_Users_M": "mean",
    "Avg_Rating": "mean",
    "Problems_Count": "mean",
    "Learning_Paths": "mean",
    "Platform_Score": "mean"
}).round(2)

platform_profiles

,Active_Users_M,Avg_Rating,Problems_Count,Learning_Paths,Platform_Score
Platform,,,,,
CodeChef,15.75,4.26,2245.25,5.41,93.78
Codeforces,15.75,4.26,2227.68,5.54,94.28
HackerRank,15.61,4.26,2234.32,5.47,93.61
LeetCode,15.49,4.25,2292.43,5.55,93.53


In [13]:
categorical_profile = df.groupby("Platform").agg({
    "Problems_Difficulty": lambda x: x.mode()[0],
    "Job_Integration": lambda x: x.mode()[0],
    "Contest_Frequency": lambda x: x.mode()[0],
    "Pricing_Model": lambda x: x.mode()[0],
    "Mobile_App_Available": lambda x: x.mode()[0],
    "Certifications_Offered": lambda x: x.mode()[0],
    "Forum_Activity_Level": lambda x: x.mode()[0]
})

categorical_profile

,Problems_Difficulty,Job_Integration,Contest_Frequency,Pricing_Model,Mobile_App_Available,Certifications_Offered,Forum_Activity_Level
Platform,,,,,,,
CodeChef,Easy,No,Occasionally,Paid,True,True,Low
Codeforces,Hard,Yes,Biweekly,Paid,True,False,High
HackerRank,Medium,No,Occasionally,Paid,False,False,Low
LeetCode,Easy,Limited,Occasionally,Paid,True,True,Medium


In [14]:
platform_profiles = platform_profiles.join(categorical_profile)

platform_profiles

,Active_Users_M,Avg_Rating,Problems_Count,Learning_Paths,Platform_Score,Problems_Difficulty,Job_Integration,Contest_Frequency,Pricing_Model,Mobile_App_Available,Certifications_Offered,Forum_Activity_Level
Platform,,,,,,,,,,,,
CodeChef,15.75,4.26,2245.25,5.41,93.78,Easy,No,Occasionally,Paid,True,True,Low
Codeforces,15.75,4.26,2227.68,5.54,94.28,Hard,Yes,Biweekly,Paid,True,False,High
HackerRank,15.61,4.26,2234.32,5.47,93.61,Medium,No,Occasionally,Paid,False,False,Low
LeetCode,15.49,4.25,2292.43,5.55,93.53,Easy,Limited,Occasionally,Paid,True,True,Medium


In [19]:
def calculate_match_score(
    user_preferences,
    platform_profile
):

    score = 0

    weights = {
        "Job_Integration": 25,
        "Contest_Frequency": 20,
        "Problems_Difficulty": 15,
        "Pricing_Model": 15,
        "Forum_Activity_Level": 15,
        "Certifications_Offered": 10
    }

    for feature, weight in weights.items():

        if user_preferences[feature] == platform_profile[feature]:
            score += weight

    return score

In [18]:
def recommend_platforms(user_preferences, platform_profiles):

    results = []

    for platform, profile in platform_profiles.iterrows():

        score = calculate_match_score(
            user_preferences,
            profile
        )

        results.append({
            "Platform": platform,
            "Match_Score": score
        })

    results = sorted(
        results,
        key=lambda x: x["Match_Score"],
        reverse=True
    )

    return results

In [21]:
user_preferences = {
    "Job_Integration": "Yes",
    "Contest_Frequency": "Biweekly",
    "Problems_Difficulty": "Hard",
    "Pricing_Model": "Paid",
    "Forum_Activity_Level": "High",
    "Certifications_Offered": False
}

In [22]:
recommend_platforms(
    user_preferences,
    platform_profiles
)

[{'Platform': 'Codeforces', 'Match_Score': 100},
 {'Platform': 'HackerRank', 'Match_Score': 25},
 {'Platform': 'CodeChef', 'Match_Score': 15},
 {'Platform': 'LeetCode', 'Match_Score': 15}]